# Inventory Demand Forecasting — Feature Engineering

In this notebook, we prepare the sales data for forecasting.

We will:
1. Select 100 product-store series.
2. Put the sales data into a simple table.
3. Add calendar information.
4. Create features from past sales.
5. Set up the forecast dates.
6. Save the files for the next notebook.

Each code cell does one small task so it is easier to follow.

## 1. Import libraries

We use:
- `pandas` to work with tables.
- `numpy` for small numerical operations.
- `Path` to build file paths.
- `json` to save our model settings.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

## 2. Find the project folders

`PROJECT_ROOT` is the main project folder. We look for the `Datasets` folder starting from the notebook's current location.

The processed folder will store the files created in this notebook.

In [2]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "Datasets").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "Datasets" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "Datasets" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Raw data folder found:", RAW_DIR.exists())

Project folder: /Users/palwaitejaswini/Inventory-demand-forecasting
Raw data folder found: True


## 3. Set our experiment settings

We use 100 series to keep the experiment manageable. The same selection will be used for every model.

Each forecast predicts seven days. We test four consecutive forecast windows.

In [3]:
N_SERIES = 100
RANDOM_STATE = 42
HORIZON = 7
N_WINDOWS = 4

## 4. Load the sales and calendar files

The sales file contains one row per product-store series, with one column for each day.

The calendar file tells us the real date and calendar information for each day.

In [4]:
sales = pd.read_csv(RAW_DIR / "sales_train_validation.csv")

calendar = pd.read_csv(
    RAW_DIR / "calendar.csv",
    parse_dates=["date"]
)

print("Sales table:", sales.shape)
print("Calendar table:", calendar.shape)

Sales table: (30490, 1919)
Calendar table: (1969, 14)


## 5. Identify the daily sales columns

Columns like `d_1`, `d_2`, and `d_3` contain the daily sales values. We store their names in a list so we can use them together.

In [5]:
sales_cols = [
    column for column in sales.columns
    if column.startswith("d_") and column[2:].isdigit()
]

print("Number of sales days:", len(sales_cols))
print("First sales day:", sales_cols[0])
print("Last sales day:", sales_cols[-1])

Number of sales days: 1913
First sales day: d_1
Last sales day: d_1913


## 6. Select 100 series across the stores

A series represents the sales history for one product at one store.

We select an equal number from each store. The fixed random seed makes the selection repeatable.

In [6]:
stores = sorted(sales["store_id"].unique())

print("Stores in the dataset:", stores)
print("Number of stores:", len(stores))

Stores in the dataset: ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']
Number of stores: 10


There are ten stores in this dataset, so we select ten series from each store to make 100 series in total.

In [7]:
series_per_store = N_SERIES // len(stores)

selected_parts = []

for store_number, store in enumerate(stores):
    store_sales = sales[sales["store_id"] == store]

    # Use a different but repeatable seed for each store.
    store_sample = store_sales.sample(
        n=series_per_store,
        random_state=RANDOM_STATE + store_number
    )

    selected_parts.append(store_sample)

selected = pd.concat(selected_parts, ignore_index=True)
selected = selected.sort_values("id").reset_index(drop=True)

print("Selected series:", selected["id"].nunique())

Selected series: 100


Let's confirm that every store has the same number of selected series.

In [8]:
display(
    selected.groupby("store_id")["id"]
    .nunique()
    .to_frame("selected series")
)

,selected series
store_id,
CA_1,10
CA_2,10
CA_3,10
CA_4,10
TX_1,10
TX_2,10
TX_3,10
WI_1,10
WI_2,10


## 7. Keep the series identifiers

These columns describe which product and store a sales row belongs to. We keep them when reshaping the data.

In [9]:
id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

display(selected[id_cols].head())

,id,item_id,dept_id,cat_id,store_id,state_id
0,FOODS_1_026_TX_3_validation,FOODS_1_026,FOODS_1,FOODS,TX_3,TX
1,FOODS_1_057_CA_2_validation,FOODS_1_057,FOODS_1,FOODS,CA_2,CA
2,FOODS_1_065_WI_1_validation,FOODS_1_065,FOODS_1,FOODS,WI_1,WI
3,FOODS_1_066_WI_2_validation,FOODS_1_066,FOODS_1,FOODS,WI_2,WI
4,FOODS_1_078_WI_1_validation,FOODS_1_078,FOODS_1,FOODS,WI_1,WI


## 8. Convert sales from wide format to long format

At the moment, each day has its own column. `melt()` turns those day columns into rows.

After this step, each row represents one product-store series on one day.

In [10]:
long = selected.melt(
    id_vars=id_cols,
    value_vars=sales_cols,
    var_name="d",
    value_name="sales"
)

display(long.head())

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,FOODS_1_026_TX_3_validation,FOODS_1_026,FOODS_1,FOODS,TX_3,TX,d_1,0
1,FOODS_1_057_CA_2_validation,FOODS_1_057,FOODS_1,FOODS,CA_2,CA,d_1,0
2,FOODS_1_065_WI_1_validation,FOODS_1_065,FOODS_1,FOODS,WI_1,WI,d_1,1
3,FOODS_1_066_WI_2_validation,FOODS_1_066,FOODS_1,FOODS,WI_2,WI,d_1,0
4,FOODS_1_078_WI_1_validation,FOODS_1_078,FOODS_1,FOODS,WI_1,WI,d_1,0


We can check the row count. There should be 100 series multiplied by 1,913 days.

In [11]:
print("Rows in long table:", len(long))
print("Expected rows:", N_SERIES * len(sales_cols))

Rows in long table: 191300
Expected rows: 191300


## 9. Prepare the calendar information

We only need the calendar columns that will be used later. Keeping a smaller table makes the merge easier to understand.

In [12]:
calendar_cols = [
    "d",
    "date",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

calendar_small = calendar[calendar_cols].copy()

display(calendar_small.head())

,d,date,wday,month,year,event_name_1,event_type_1,snap_CA,snap_TX,snap_WI
0,d_1,2011-01-29,1,1,2011,NaN,NaN,0,0,0
1,d_2,2011-01-30,2,1,2011,NaN,NaN,0,0,0
2,d_3,2011-01-31,3,1,2011,NaN,NaN,0,0,0
3,d_4,2011-02-01,4,2,2011,NaN,NaN,1,1,0
4,d_5,2011-02-02,5,2,2011,NaN,NaN,1,0,1


## 10. Add the calendar to the sales table

The column `d` appears in both tables, so we use it to match each sales day to its calendar date.

`how="left"` keeps every sales row.

In [13]:
long = long.merge(
    calendar_small,
    on="d",
    how="left",
    validate="many_to_one"
)

long = long.sort_values(["id", "date"]).reset_index(drop=True)

display(long[["id", "d", "date", "sales", "month"]].head())

,id,d,date,sales,month
0,FOODS_1_026_TX_3_validation,d_1,2011-01-29,0,1
1,FOODS_1_026_TX_3_validation,d_2,2011-01-30,0,1
2,FOODS_1_026_TX_3_validation,d_3,2011-01-31,0,1
3,FOODS_1_026_TX_3_validation,d_4,2011-02-01,1,2
4,FOODS_1_026_TX_3_validation,d_5,2011-02-02,0,2


## 11. Create an event feature

`event` is 1 when the calendar lists an event and 0 when no event name is listed.

In [14]:
long["event"] = long["event_name_1"].notna().astype("int8")

display(long[["date", "event_name_1", "event"]].head(10))

,date,event_name_1,event
0,2011-01-29,NaN,0
1,2011-01-30,NaN,0
2,2011-01-31,NaN,0
3,2011-02-01,NaN,0
4,2011-02-02,NaN,0
5,2011-02-03,NaN,0
6,2011-02-04,NaN,0
7,2011-02-05,NaN,0
8,2011-02-06,SuperBowl,1
9,2011-02-07,NaN,0


## 12. Create weekday and weekend features

`day_of_week` uses numbers from 0 to 6: Monday is 0 and Sunday is 6.

`is_weekend` is 1 for Saturday and Sunday, and 0 for the other days.

In [15]:
long["day_of_week"] = long["date"].dt.dayofweek.astype("int8")

long["is_weekend"] = (
    long["day_of_week"].isin([5, 6])
).astype("int8")

display(long[["date", "day_of_week", "is_weekend"]].head(10))

,date,day_of_week,is_weekend
0,2011-01-29,5,1
1,2011-01-30,6,1
2,2011-01-31,0,0
3,2011-02-01,1,0
4,2011-02-02,2,0
5,2011-02-03,3,0
6,2011-02-04,4,0
7,2011-02-05,5,1
8,2011-02-06,6,1
9,2011-02-07,0,0


## 13. Create the SNAP feature

The calendar has a separate SNAP indicator for California, Texas, and Wisconsin. We select the indicator that matches the series' state.

In [16]:
long["snap"] = np.select(
    [
        long["state_id"].eq("CA"),
        long["state_id"].eq("TX"),
        long["state_id"].eq("WI")
    ],
    [
        long["snap_CA"],
        long["snap_TX"],
        long["snap_WI"]
    ],
    default=0
).astype("int8")

display(long[["state_id", "date", "snap"]].head(10))

,state_id,date,snap
0,TX,2011-01-29,0
1,TX,2011-01-30,0
2,TX,2011-01-31,0
3,TX,2011-02-01,1
4,TX,2011-02-02,0
5,TX,2011-02-03,1
6,TX,2011-02-04,0
7,TX,2011-02-05,1
8,TX,2011-02-06,1
9,TX,2011-02-07,1


## 14. Create numeric item and store codes

The model needs numeric inputs for item and store identity. These codes are labels only; a larger code does not mean a larger or better item.

In [17]:
long["item_code"] = pd.Categorical(long["item_id"]).codes.astype("int16")
long["store_code"] = pd.Categorical(long["store_id"]).codes.astype("int8")

display(long[["item_id", "item_code", "store_id", "store_code"]].drop_duplicates().head(10))

,item_id,item_code,store_id,store_code
0,FOODS_1_026,0,TX_3,6
1913,FOODS_1_057,1,CA_2,1
3826,FOODS_1_065,2,WI_1,7
5739,FOODS_1_066,3,WI_2,8
7652,FOODS_1_078,4,WI_1,7
9565,FOODS_1_092,5,TX_1,4
11478,FOODS_2_088,6,CA_4,3
13391,FOODS_2_135,7,CA_1,0
15304,FOODS_2_139,8,WI_3,9
17217,FOODS_2_144,9,WI_3,9


## 15. Sort the data before using past sales

Lag features only make sense when each series is in date order. We sort by series and date before creating them.

In [18]:
long = long.sort_values(["id", "date"]).reset_index(drop=True)

print("Rows:", len(long))
print("Series:", long["id"].nunique())

Rows: 191300
Series: 100


## 16. Create the one-day lag

`lag_1` is the previous day's sales for the same product-store series.

`groupby("id")` keeps the history of each series separate. `shift(1)` moves the sales values down by one row.

In [19]:
long["lag_1"] = long.groupby("id")["sales"].shift(1)

display(long[["id", "date", "sales", "lag_1"]].head(10))

,id,date,sales,lag_1
0,FOODS_1_026_TX_3_validation,2011-01-29,0,NaN
1,FOODS_1_026_TX_3_validation,2011-01-30,0,0.0
2,FOODS_1_026_TX_3_validation,2011-01-31,0,0.0
3,FOODS_1_026_TX_3_validation,2011-02-01,1,0.0
4,FOODS_1_026_TX_3_validation,2011-02-02,0,1.0
5,FOODS_1_026_TX_3_validation,2011-02-03,0,0.0
6,FOODS_1_026_TX_3_validation,2011-02-04,0,0.0
7,FOODS_1_026_TX_3_validation,2011-02-05,1,0.0
8,FOODS_1_026_TX_3_validation,2011-02-06,0,1.0
9,FOODS_1_026_TX_3_validation,2011-02-07,0,0.0


## 17. Create the seven-day lag

`lag_7` is the sales value from seven days earlier. It can help the model learn weekly patterns.

In [20]:
long["lag_7"] = long.groupby("id")["sales"].shift(7)

display(long[["id", "date", "sales", "lag_7"]].head(12))

,id,date,sales,lag_7
0,FOODS_1_026_TX_3_validation,2011-01-29,0,NaN
1,FOODS_1_026_TX_3_validation,2011-01-30,0,NaN
2,FOODS_1_026_TX_3_validation,2011-01-31,0,NaN
3,FOODS_1_026_TX_3_validation,2011-02-01,1,NaN
4,FOODS_1_026_TX_3_validation,2011-02-02,0,NaN
5,FOODS_1_026_TX_3_validation,2011-02-03,0,NaN
6,FOODS_1_026_TX_3_validation,2011-02-04,0,NaN
7,FOODS_1_026_TX_3_validation,2011-02-05,1,0.0
8,FOODS_1_026_TX_3_validation,2011-02-06,0,0.0
9,FOODS_1_026_TX_3_validation,2011-02-07,0,0.0


## 18. Create the 28-day lag

`lag_28` is the sales value from 28 days earlier. The first 28 days of each series do not have enough history for this feature.

In [21]:
long["lag_28"] = long.groupby("id")["sales"].shift(28)

display(long[["id", "date", "sales", "lag_28"]].head(32))

,id,date,sales,lag_28
0,FOODS_1_026_TX_3_validation,2011-01-29,0,NaN
1,FOODS_1_026_TX_3_validation,2011-01-30,0,NaN
2,FOODS_1_026_TX_3_validation,2011-01-31,0,NaN
3,FOODS_1_026_TX_3_validation,2011-02-01,1,NaN
4,FOODS_1_026_TX_3_validation,2011-02-02,0,NaN
5,FOODS_1_026_TX_3_validation,2011-02-03,0,NaN
6,FOODS_1_026_TX_3_validation,2011-02-04,0,NaN
7,FOODS_1_026_TX_3_validation,2011-02-05,1,NaN
8,FOODS_1_026_TX_3_validation,2011-02-06,0,NaN
9,FOODS_1_026_TX_3_validation,2011-02-07,0,NaN


## 19. Create the seven-day rolling average

We first shift sales by one day, then calculate the average of the previous seven values.

The shift is important: it stops today's actual sales from being used to predict today's sales.

In [22]:
long["rolling_mean_7"] = (
    long.groupby("id")["sales"]
    .transform(lambda values: values.shift(1).rolling(7).mean())
)

display(long[["id", "date", "sales", "rolling_mean_7"]].head(12))

,id,date,sales,rolling_mean_7
0,FOODS_1_026_TX_3_validation,2011-01-29,0,NaN
1,FOODS_1_026_TX_3_validation,2011-01-30,0,NaN
2,FOODS_1_026_TX_3_validation,2011-01-31,0,NaN
3,FOODS_1_026_TX_3_validation,2011-02-01,1,NaN
4,FOODS_1_026_TX_3_validation,2011-02-02,0,NaN
5,FOODS_1_026_TX_3_validation,2011-02-03,0,NaN
6,FOODS_1_026_TX_3_validation,2011-02-04,0,NaN
7,FOODS_1_026_TX_3_validation,2011-02-05,1,0.142857
8,FOODS_1_026_TX_3_validation,2011-02-06,0,0.285714
9,FOODS_1_026_TX_3_validation,2011-02-07,0,0.285714


## 20. Create the 28-day rolling average

This is the same idea, but it averages the previous 28 days instead of seven.

In [23]:
long["rolling_mean_28"] = (
    long.groupby("id")["sales"]
    .transform(lambda values: values.shift(1).rolling(28).mean())
)

display(long[["id", "date", "sales", "rolling_mean_28"]].head(32))

,id,date,sales,rolling_mean_28
0,FOODS_1_026_TX_3_validation,2011-01-29,0,NaN
1,FOODS_1_026_TX_3_validation,2011-01-30,0,NaN
2,FOODS_1_026_TX_3_validation,2011-01-31,0,NaN
3,FOODS_1_026_TX_3_validation,2011-02-01,1,NaN
4,FOODS_1_026_TX_3_validation,2011-02-02,0,NaN
5,FOODS_1_026_TX_3_validation,2011-02-03,0,NaN
6,FOODS_1_026_TX_3_validation,2011-02-04,0,NaN
7,FOODS_1_026_TX_3_validation,2011-02-05,1,NaN
8,FOODS_1_026_TX_3_validation,2011-02-06,0,NaN
9,FOODS_1_026_TX_3_validation,2011-02-07,0,NaN


## 21. List the features for the forecasting model

These are the columns the Gradient Boosting model can use as inputs. The target it learns to predict is `sales`.

In [24]:
FEATURES = [
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "is_weekend",
    "event",
    "snap",
    "item_code",
    "store_code"
]

print("Number of features:", len(FEATURES))
print(FEATURES)

Number of features: 12
['lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'month', 'is_weekend', 'event', 'snap', 'item_code', 'store_code']


## 22. Check that the lag features are correct

We choose one series and one date, then compare its `lag_1` and `lag_7` values with the actual earlier sales.

These checks help catch mistakes in the feature calculations.

In [25]:
first_id = long["id"].iloc[0]
one_series = long[long["id"] == first_id].reset_index(drop=True)

row_number = 40
row = one_series.iloc[row_number]

print("Current date:", row["date"])
print("Current sales:", row["sales"])
print("lag_1:", row["lag_1"], "| previous sales:", one_series.iloc[row_number - 1]["sales"])
print("lag_7:", row["lag_7"], "| sales seven rows earlier:", one_series.iloc[row_number - 7]["sales"])

Current date: 2011-03-10 00:00:00
Current sales: 0
lag_1: 0.0 | previous sales: 0
lag_7: 1.0 | sales seven rows earlier: 1


The rolling average should also match the average of the seven sales values before the selected date.

In [26]:
previous_seven_sales = one_series.iloc[row_number - 7:row_number]["sales"].mean()

print("Stored rolling average:", row["rolling_mean_7"])
print("Calculated from past sales:", previous_seven_sales)

assert np.isclose(row["rolling_mean_7"], previous_seven_sales)
print("Rolling average check passed.")

Stored rolling average: 0.7142857142857143
Calculated from past sales: 0.7142857142857143
Rolling average check passed.


## 23. Check missing values in the features

Lag and rolling features are expected to be missing at the start of each series because there is not enough earlier history. The calendar features should be available.

In [27]:
display(long[FEATURES].isna().sum().to_frame("missing values"))

,missing values
lag_1,100
lag_7,700
lag_28,2800
rolling_mean_7,700
rolling_mean_28,2800
day_of_week,0
month,0
is_weekend,0
event,0
snap,0


## 24. Set the four forecast windows

We use the final 28 days of the available sales history for evaluation. Each window covers seven days.

The forecast origin is the day immediately before that window starts.

In [28]:
all_dates = pd.DatetimeIndex(sorted(long["date"].unique()))

validation_dates = all_dates[-HORIZON * N_WINDOWS:]

print("First evaluation date:", validation_dates[0])
print("Last evaluation date:", validation_dates[-1])

First evaluation date: 2016-03-28 00:00:00
Last evaluation date: 2016-04-24 00:00:00


Split those 28 dates into four consecutive windows. We store the start date, end date, and forecast origin for each one.

In [29]:
windows = []

for window_number in range(N_WINDOWS):
    start_position = window_number * HORIZON
    end_position = start_position + HORIZON

    dates = validation_dates[start_position:end_position]

    windows.append({
        "window": window_number + 1,
        "origin": dates[0] - pd.Timedelta(days=1),
        "forecast_start": dates[0],
        "forecast_end": dates[-1]
    })

windows = pd.DataFrame(windows)

display(windows)

,window,origin,forecast_start,forecast_end
0,1,2016-03-27,2016-03-28,2016-04-03
1,2,2016-04-03,2016-04-04,2016-04-10
2,3,2016-04-10,2016-04-11,2016-04-17
3,4,2016-04-17,2016-04-18,2016-04-24


Check that every forecast window is exactly seven days long and starts the day after its origin.

In [30]:
assert (windows["forecast_start"] == windows["origin"] + pd.Timedelta(days=1)).all()
assert (
    windows["forecast_end"] - windows["forecast_start"]
    == pd.Timedelta(days=6)
).all()

print("All four forecast windows are valid.")

All four forecast windows are valid.


## 25. Save the prepared data

Notebook 3 will load these files so all models use the same series and forecast windows.

We save:
- The sales table with features.
- The selected series IDs.
- The forecast windows.
- The settings used in this notebook.

In [31]:
columns_to_save = id_cols + [
    "d", "date", "sales",
    "day_of_week", "month", "is_weekend", "event", "snap",
    "item_code", "store_code",
    "lag_1", "lag_7", "lag_28",
    "rolling_mean_7", "rolling_mean_28"
]

prepared_data = long[columns_to_save].copy()

prepared_data.to_pickle(PROCESSED_DIR / "phase1_prepared.pkl")

selected[id_cols].to_csv(
    PROCESSED_DIR / "phase1_selected_series.csv",
    index=False
)

windows.to_csv(
    PROCESSED_DIR / "phase1_forecast_windows.csv",
    index=False
)

config = {
    "n_series": N_SERIES,
    "random_state": RANDOM_STATE,
    "horizon": HORIZON,
    "n_windows": N_WINDOWS,
    "features": FEATURES
}

(PROCESSED_DIR / "phase1_config.json").write_text(
    json.dumps(config, indent=2)
)

print("Saved files to:", PROCESSED_DIR)

Saved files to: /Users/palwaitejaswini/Inventory-demand-forecasting/Datasets/processed


## 26. Final checks

Before moving to the model notebook, check the size of the prepared table and confirm that the expected files were saved.

In [32]:
expected_rows = N_SERIES * len(sales_cols)
expected_missing = {
    "lag_1": N_SERIES,
    "lag_7": N_SERIES * 7,
    "lag_28": N_SERIES * 28,
    "rolling_mean_7": N_SERIES * 7,
    "rolling_mean_28": N_SERIES * 28
}

assert len(prepared_data) == expected_rows
assert prepared_data["id"].nunique() == N_SERIES
assert prepared_data["date"].notna().all()
assert not prepared_data[["id", "date"]].duplicated().any()
assert prepared_data.groupby("id").size().eq(len(sales_cols)).all()
assert prepared_data.groupby("id")["date"].apply(lambda dates: dates.is_monotonic_increasing).all()
assert selected.groupby("store_id")["id"].nunique().eq(N_SERIES // len(stores)).all()

for feature, missing_count in expected_missing.items():
    assert prepared_data[feature].isna().sum() == missing_count

output_files = [
    PROCESSED_DIR / "phase1_prepared.pkl",
    PROCESSED_DIR / "phase1_selected_series.csv",
    PROCESSED_DIR / "phase1_forecast_windows.csv",
    PROCESSED_DIR / "phase1_config.json"
]
assert all(path.exists() for path in output_files)

print("Prepared rows:", len(prepared_data))
print("Prepared series:", prepared_data["id"].nunique())
print("Forecast windows:", len(windows))
print("Duplicate series-date keys:", prepared_data[["id", "date"]].duplicated().sum())
print("All final data and output checks passed.")

Prepared rows: 191300


Prepared series: 100
Forecast windows: 4
Duplicate series-date keys: 0
All final data and output checks passed.


## What we learned

- Each row now represents one product-store series on one date.
- Calendar features describe the date and known events.
- Lag features use sales from earlier days.
- Rolling averages summarize recent sales without using the current day's actual sales.
- The final 28 days are split into four seven-day forecast windows.
- The prepared files are saved for the next notebook.

**Important:** During a multi-day forecast, the model must use its own predictions to update lag features for later days in the same window. It must not use the hidden actual sales from those forecast days.